# CreditWise — Exploratory Data Analysis

**Notebook 01 of 04**

This notebook investigates the Give Me Some Credit dataset to understand:
- Shape, dtypes, missing values, duplicates
- Target class distribution and imbalance
- Feature distributions and outliers
- Correlations between features
- Relationship between features and the default target
- Data quality issues that must be addressed in preprocessing

> **Dataset**: cs-training.csv — Give Me Some Credit (Kaggle)
> 
> **Target**: `SeriousDlqin2yrs` — 0 = no serious delinquency, 1 = serious delinquency within 2 years

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

from src.data_loader import load_raw_data, get_dataset_summary
from src.config import TARGET_COLUMN, FIGURES_DIR

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',
    'grid.color': '#334155',
    'grid.alpha': 0.5,
    'figure.dpi': 110,
})
PALETTE = ['#38bdf8', '#f43f5e']
print('Imports complete.')

## 1. Load and Inspect the Dataset

In [ ]:
df = load_raw_data()
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
display(df.head())

In [ ]:
summary = get_dataset_summary(df)
print('--- Dataset Summary ---')
for k, v in summary.items():
    print(f'  {k:30s}: {v}')

In [ ]:
print('--- Data Types ---')
display(df.dtypes.to_frame(name='dtype'))
print()
print('--- Descriptive Statistics ---')
display(df.describe().round(3).T)

## 2. Missing Values

Two features have missing values:
- `MonthlyIncome`: ~19.8% missing
- `NumberOfDependents`: ~2.6% missing

Treatment: **Median imputation** inside the sklearn Pipeline (applied to training data only, then propagated to test/inference).

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_display = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
display(missing_display[missing_display['Missing Count'] > 0])

In [ ]:
# Check: is missingness in MonthlyIncome related to default?
df['income_missing'] = df['MonthlyIncome'].isna().astype(int)
miss_default = df.groupby('income_missing')[TARGET_COLUMN].mean() * 100
print('Default rate by MonthlyIncome missingness:')
print(miss_default.rename({0: 'Income Present', 1: 'Income Missing'}).round(2), '%')
df = df.drop(columns=['income_missing'])

## 3. Duplicate Rows

In [ ]:
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup/len(df)*100:.2f}%)')
print('Decision: Drop duplicates during preprocessing (before splitting).')

## 4. Target Class Distribution — Class Imbalance

In [ ]:
counts = df[TARGET_COLUMN].value_counts().sort_index()
pcts = df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100
print('Target distribution:')
for v in [0, 1]:
    print(f'  Class {v}: {counts[v]:,}  ({pcts[v]:.2f}%)')
print(f'  Imbalance ratio: {counts[0]/counts[1]:.2f} : 1')
print()
print('Implication: Simple accuracy is misleading.')
print('A model predicting ALL zeros would achieve 93.3% accuracy — but catch zero defaults.')
print('Primary metric: ROC-AUC + PR-AUC + Recall for the positive class.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels = ['No Default (0)', 'Default (1)']
bars = axes[0].bar(labels, counts.values, color=PALETTE, width=0.5, edgecolor='#0f172a')
axes[0].set_title('Target Class Counts')
axes[0].set_ylabel('Number of Borrowers')
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+400,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].grid(axis='y', alpha=0.4)

axes[1].pie(counts.values, labels=labels, colors=PALETTE, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'#0f172a','linewidth':2},
            textprops={'color':'#e2e8f0'})
axes[1].set_title('Target Class Proportion')
plt.suptitle('SeriousDlqin2yrs — Severe Class Imbalance (13.96:1)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Feature Distributions (Histograms)

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COLUMN]
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df[col].dropna()
    data_log = np.log1p(data.clip(lower=0))
    ax.hist(data_log, bins=60, color='#38bdf8', edgecolor='#0f172a', alpha=0.85)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('log1p scale', fontsize=7)
    ax.axvline(data_log.mean(), color='#f59e0b', linestyle='--', lw=1.5)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Feature Distributions (log1p transformed for skewed features)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Outlier Analysis

In [ ]:
# Extreme value summary
outlier_summary = []
for col in feature_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    outlier_summary.append({
        'Feature': col, 'Q1': round(q1,3), 'Q3': round(q3,3),
        'IQR': round(iqr,3), 'Max': round(s.max(),2),
        'IQR Outliers': n_outliers,
        'IQR Outlier%': round(n_outliers/len(s)*100,2),
        'Skewness': round(s.skew(),2)
    })
display(pd.DataFrame(outlier_summary))

In [ ]:
# Specific data quality flags
print('=== DATA QUALITY FLAGS ===')
print(f"age == 0: {(df['age']==0).sum()} rows (should be impossible)")
print(f"RevolvingUtilization > 1: {(df['RevolvingUtilizationOfUnsecuredLines']>1).sum():,} rows ({(df['RevolvingUtilizationOfUnsecuredLines']>1).mean()*100:.1f}%)")
print(f"DebtRatio > 100: {(df['DebtRatio']>100).sum():,} rows ({(df['DebtRatio']>100).mean()*100:.2f}%)")
for col in ['NumberOfTime30-59DaysPastDueNotWorse',
            'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate']:
    n96 = (df[col]==96).sum()
    n98 = (df[col]==98).sum()
    print(f"{col}: value=96 → {n96}, value=98 → {n98} (likely missing-value codes)")

## 7. Correlation Heatmap

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5, linecolor='#0f172a',
            annot_kws={'size':8}, ax=ax, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Heatmap (Pearson)', fontsize=13)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=abs, ascending=False)
print('\nTop feature correlations with SeriousDlqin2yrs:')
print(target_corr.round(4))

## 8. Default Rate by Feature Quartile

In [ ]:
key_features = ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate',
                'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'age']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    temp = df[[col, TARGET_COLUMN]].dropna().copy()
    try:
        temp['bin'] = pd.qcut(temp[col], q=4, duplicates='drop')
    except ValueError:
        temp['bin'] = temp[col].clip(upper=10)

    dr = temp.groupby('bin', observed=True)[TARGET_COLUMN].mean() * 100
    bars = ax.bar(range(len(dr)), dr.values, color='#38bdf8', edgecolor='#0f172a', alpha=0.85)
    ax.set_xticks(range(len(dr)))
    ax.set_xticklabels([str(b) for b in dr.index], fontsize=7, rotation=30, ha='right')
    ax.axhline(df[TARGET_COLUMN].mean()*100, color='#f59e0b', linestyle='--',
               lw=1.5, label=f'Overall: {df[TARGET_COLUMN].mean()*100:.1f}%')
    ax.set_title(f'Default Rate by {col}', fontsize=9)
    ax.set_ylabel('Default Rate (%)', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Observed Default Rate by Feature Quartile', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Feature Distributions: Defaulters vs Non-Defaulters

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    for t_val, color, lbl in zip([0, 1], PALETTE, ['No Default', 'Default']):
        subset = df.loc[df[TARGET_COLUMN]==t_val, col].dropna()
        subset_log = np.log1p(subset.clip(lower=0))
        ax.hist(subset_log, bins=50, alpha=0.5, color=color, label=lbl,
                density=True, edgecolor='none')
        if subset_log.nunique() > 5:
            try:
                kde = gaussian_kde(subset_log)
                xs = np.linspace(subset_log.min(), subset_log.max(), 200)
                ax.plot(xs, kde(xs), color=color, lw=2)
            except Exception:
                pass
    ax.set_title(f'{col} (log1p)', fontsize=9)
    ax.set_xlabel('log1p value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Feature Distributions: Defaulters vs Non-Defaulters', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Age and Income Deep Dives

In [ ]:
# Age bracket default rates
bins = [0, 30, 40, 50, 60, 70, 120]
age_labels = ['<30', '30-40', '40-50', '50-60', '60-70', '70+']
df_age = df[['age', TARGET_COLUMN]].dropna().copy()
df_age['age_group'] = pd.cut(df_age['age'], bins=bins, labels=age_labels, right=False)
age_stats = df_age.groupby('age_group', observed=True).agg(
    count=(TARGET_COLUMN, 'count'),
    default_rate=(TARGET_COLUMN, 'mean')
).round(4)
age_stats['default_rate_pct'] = (age_stats['default_rate']*100).round(2)
display(age_stats)

# Min age issue
print(f"\nrows with age==0: {(df['age']==0).sum()}")
print(f"rows with age < 18: {(df['age']<18).sum()}")

In [ ]:
# Income stats by class
income_stats = df.groupby(TARGET_COLUMN)['MonthlyIncome'].describe().round(2)
display(income_stats.rename({0: 'No Default', 1: 'Default'}))

## 11. EDA Key Findings Summary

In [ ]:
findings = [
    ('Class Imbalance', f"93.3% class 0 vs 6.7% class 1. Ratio = 13.96:1. Primary metrics: ROC-AUC, PR-AUC, Recall."),
    ('Missing Values', "MonthlyIncome: 19.82% missing. NumberOfDependents: 2.62% missing. Treatment: median imputation."),
    ('Duplicate Rows', "609 duplicate rows (~0.4%). Drop before splitting."),
    ('Extreme Outliers — RevolvingUtil', f"{(df['RevolvingUtilizationOfUnsecuredLines']>1).mean()*100:.1f}% of values > 1.0. Log-transform justified."),
    ('Extreme Outliers — DebtRatio', f"{(df['DebtRatio']>100).mean()*100:.2f}% values > 100. Likely data errors, not real debt."),
    ('Delinquency Codes', "Values 96/98 in past-due columns likely represent missing codes. Treat as outliers."),
    ('Age = 0', f"{(df['age']==0).sum()} rows with age=0. Likely errors; will be handled by validation."),
    ('High Skewness', "MonthlyIncome, DebtRatio, RevolvingUtil are severely right-skewed → log transforms."),
    ('Top Correlated Feature', f"NumberOfTimes90DaysLate has highest abs correlation with target."),
    ('Low Multicollinearity', "Delinquency columns moderately correlated with each other; others are low."),
]

print('=== EDA KEY FINDINGS ===')
for name, finding in findings:
    print(f'\n[{name}]')
    print(f'  {finding}')

## 12. Preprocessing Decisions Driven by EDA

| Finding | Preprocessing Decision |
|---|---|
| MonthlyIncome 19.8% missing | Median imputation (sklearn Pipeline) |
| NumberOfDependents 2.6% missing | Median imputation |
| 609 duplicate rows | Drop duplicates before split |
| RevolvingUtilization skewed + outliers | log1p transform (feature engineering) |
| DebtRatio extreme values | log1p transform (feature engineering) |
| MonthlyIncome skewed | log1p transform (feature engineering) |
| Delinquency values 96/98 | Treat as outliers; log transform reduces impact |
| Age = 0 | Input validation in the inference module |
| Class imbalance 13.96:1 | Compare class_weight, SMOTE, random oversampling |
| Logistic Regression | Requires StandardScaler (included in pipeline) |